<a href="https://colab.research.google.com/github/isabelamakis-hash/7_tentavita-de-carregar_essa_Landing_page/blob/main/voice_chat_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Voice Chat com ChatGPT + Whisper (PT-BR)
> **Google Colab Edition** — grave sua voz, transcreva com Whisper e ouça a resposta do GPT-4o!

### Fluxo:
```
🎤 Microfone  →  🗣️ Whisper (transcrição)  →  🤖 GPT-4o (resposta)  →  🔊 TTS (voz)
```

---
⚠️ **Importante:** O Colab usa JavaScript para acessar o microfone do seu navegador. Permita o acesso quando solicitado.

## 📦 Célula 1 — Instalar dependências

In [ ]:
# Instala as bibliotecas necessárias
!pip install -q openai pydub
print('✅ Dependências instaladas!')

✅ Dependências instaladas!


## 🔑 Célula 2 — Configurar API Key da OpenAI

In [ ]:
import os
from getpass import getpass

# ─── OPÇÃO A: digitar manualmente (mais seguro) ───
api_key = getpass('🔑 Cole sua OpenAI API Key (não aparece na tela): ')
os.environ['OPENAI_API_KEY'] = api_key

# ─── OPÇÃO B: usar Google Colab Secrets (recomendado) ───
# Vá em: 🔑 Secrets (painel esquerdo) → adicione OPENAI_API_KEY
# Depois descomente as linhas abaixo e comente a OPÇÃO A:
#
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

print('✅ API Key configurada!')

🔑 Cole sua OpenAI API Key (não aparece na tela): ··········
✅ API Key configurada!


## ⚙️ Célula 3 — Configurações do projeto

In [ ]:
from openai import OpenAI

# ── Configurações ──────────────────────────────────
GPT_MODEL    = 'gpt-4o'   # Modelo do ChatGPT
TTS_VOICE    = 'nova'     # Vozes: nova | alloy | echo | fable | onyx | shimmer
WHISPER_LANG = 'pt'       # Idioma para o Whisper

SYSTEM_PROMPT = """Você é um assistente virtual inteligente e amigável.
Responda SEMPRE em português do Brasil.
Seja direto, claro e conciso. Quando não souber algo, diga honestamente."""

# ── Inicialização ──────────────────────────────────
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
conversation_history = []   # memória da conversa

print('✅ Configurações carregadas!')
print(f'   Modelo : {GPT_MODEL}')
print(f'   Voz TTS: {TTS_VOICE}')

✅ Configurações carregadas!
   Modelo : gpt-4o
   Voz TTS: nova


## 🎤 Célula 4 — Funções de gravação de áudio (via JavaScript)

In [ ]:
# ─────────────────────────────────────────────────────────
# 🎤 CÉLULA 4 CORRIGIDA — Gravação de áudio no Google Colab
# Substitua a Célula 4 do seu notebook por este código!
# ─────────────────────────────────────────────────────────

from IPython.display import Javascript, display, Audio
from google.colab import output
from google.colab.output import eval_js
import base64, tempfile, os, time

def record_audio_colab(duration_seconds=8):
    """
    Grava áudio pelo microfone do navegador usando eval_js (mais confiável no Colab).
    Retorna o caminho do arquivo de áudio gravado.
    """

    js_code = f"""
    async function recordAudio() {{
        const duration = {duration_seconds * 1000};

        let stream;
        try {{
            stream = await navigator.mediaDevices.getUserMedia({{ audio: true, video: false }});
        }} catch(err) {{
            return 'ERRO_MICROFONE: ' + err.message;
        }}

        const mediaRecorder = new MediaRecorder(stream, {{ mimeType: 'audio/webm' }});
        const chunks = [];

        mediaRecorder.ondataavailable = e => {{
            if (e.data.size > 0) chunks.push(e.data);
        }};

        mediaRecorder.start(100); // coleta a cada 100ms

        await new Promise(r => setTimeout(r, duration));
        mediaRecorder.stop();

        await new Promise(r => mediaRecorder.onstop = r);
        stream.getTracks().forEach(t => t.stop());

        const blob = new Blob(chunks, {{ type: 'audio/webm' }});

        // Converte para base64
        const toBase64 = blob => new Promise((resolve, reject) => {{
            const reader = new FileReader();
            reader.onload = () => resolve(reader.result.split(',')[1]);
            reader.onerror = reject;
            reader.readAsDataURL(blob);
        }});

        const b64 = await toBase64(blob);
        return b64;
    }}

    recordAudio();
    """

    print(f'🎙️  Gravando por {duration_seconds} segundos... Fale agora!')

    # eval_js aguarda o JavaScript terminar e retorna o resultado direto
    result = eval_js(js_code)

    if not result or result.startswith('ERRO_MICROFONE'):
        erro = result if result else 'Nenhum dado retornado'
        raise RuntimeError(f'❌ Erro no microfone: {erro}\n'
                           f'   → Verifique a permissão do microfone no navegador.')

    print('✅ Gravação concluída!')

    # Salva o áudio em arquivo temporário
    audio_bytes = base64.b64decode(result)
    tmp = tempfile.NamedTemporaryFile(suffix='.webm', delete=False)
    tmp.write(audio_bytes)
    tmp.close()

    return tmp.name


print('✅ Célula 4 corrigida e pronta!')

✅ Célula 4 corrigida e pronta!


## 📝 Célula 5 — Funções de transcrição, chat e TTS

In [ ]:
import tempfile, os
from IPython.display import Audio, display


def transcribe_audio(audio_path: str) -> str:
    """Transcreve o áudio usando Whisper (OpenAI API)."""
    print('📝 Transcrevendo com Whisper...')
    with open(audio_path, 'rb') as f:
        transcript = client.audio.transcriptions.create(
            model='whisper-1',
            file=f,
            language=WHISPER_LANG,
        )
    text = transcript.text.strip()
    print(f'🗣️  Você disse: "{text}"')
    return text


def chat_with_gpt(user_message: str) -> str:
    """Envia mensagem ao GPT-4o e retorna a resposta, mantendo histórico."""
    print('🤖 Gerando resposta...')
    conversation_history.append({'role': 'user', 'content': user_message})

    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + conversation_history

    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=500,
    )

    reply = response.choices[0].message.content.strip()
    conversation_history.append({'role': 'assistant', 'content': reply})

    print(f'💬 Assistente: {reply}')
    return reply


def speak(text: str):
    """Converte texto em fala (TTS) e reproduz no Colab."""
    print('🔊 Sintetizando voz...')
    response = client.audio.speech.create(
        model='tts-1',
        voice=TTS_VOICE,
        input=text,
        response_format='mp3',
    )
    tmp = tempfile.NamedTemporaryFile(suffix='.mp3', delete=False)
    tmp.write(response.content)
    tmp.close()

    # Reproduz o áudio diretamente no Colab
    display(Audio(tmp.name, autoplay=True))
    os.unlink(tmp.name)


print('✅ Funções de transcrição, chat e TTS prontas!')

✅ Funções de transcrição, chat e TTS prontas!


## 🚀 Célula 6 — Iniciar conversa por voz!

> **Execute esta célula quantas vezes quiser** — cada execução é um turno da conversa.
>
> Ajuste `DURACAO_GRAVACAO` conforme necessário.

In [ ]:
# ─── Ajuste o tempo de gravação (em segundos) ───────
DURACAO_GRAVACAO = 8  # segundos
# ────────────────────────────────────────────────────

print('=' * 52)
print('  🎙️  Voice Chat — Turno da conversa')
print('=' * 52)

try:
    # 1. Grava o áudio do microfone
    audio_path = record_audio_colab(duration_seconds=DURACAO_GRAVACAO)

    # 2. Transcreve com Whisper
    user_text = transcribe_audio(audio_path)
    os.unlink(audio_path)  # limpa arquivo temporário

    if not user_text:
        print('⚠️  Nenhuma fala detectada. Execute novamente e fale mais alto.')
    else:
        # 3. Gera resposta com GPT-4o
        response_text = chat_with_gpt(user_text)

        # 4. Reproduz a resposta em voz
        speak(response_text)

        print(f'\n📊 Mensagens no histórico: {len(conversation_history)}')

except Exception as e:
    print(f'❌ Erro: {e}')

  🎙️  Voice Chat — Turno da conversa
🎙️  Gravando por 8 segundos... Fale agora!
✅ Gravação concluída!
📝 Transcrevendo com Whisper...
❌ Erro: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


## 🗑️ Célula 7 — Limpar histórico da conversa (opcional)

In [ ]:
conversation_history.clear()
print('🗑️  Histórico limpo! A próxima conversa começa do zero.')

🗑️  Histórico limpo! A próxima conversa começa do zero.


## 📋 Célula 8 — Ver histórico completo (opcional)

In [ ]:
if not conversation_history:
    print('📭 Histórico vazio.')
else:
    print(f'📜 Histórico — {len(conversation_history)} mensagem(ns)\n')
    print('─' * 50)
    for i, msg in enumerate(conversation_history, 1):
        role = '🗣️  Você' if msg['role'] == 'user' else '🤖 Assistente'
        print(f'{i}. {role}:')
        print(f'   {msg["content"]}')
        print('─' * 50)

📭 Histórico vazio.


---
## 💡 Dicas e resolução de problemas

| Problema | Solução |
|---|---|
| Microfone não funciona | Permita o acesso no navegador (cadeado na barra de endereço) |
| Áudio não reconhecido | Aumente `DURACAO_GRAVACAO` ou fale mais alto |
| Erro de API Key | Verifique se a chave está correta na Célula 2 |
| Resposta em outro idioma | O `SYSTEM_PROMPT` força PT-BR, mas verifique `WHISPER_LANG = 'pt'` |
| Mudar voz do assistente | Altere `TTS_VOICE` na Célula 3: `nova`, `alloy`, `echo`, `fable`, `onyx`, `shimmer` |